In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [ ]:
reporter_with_economy_142 = pd.read_csv(DATA / 'reporter-with-economy-142.csv')

In [ ]:
df_production = pd.read_csv(DATA / 'production-142.csv', index_col=0)
economies_142 = df_production.columns.tolist()

In [4]:
def aggregate_matrix_to_142(lm):
    mapping = reporter_with_economy_142
    target_order = economies_142
    
    # code -> econ label (only non-empty mappings)
    map_series = mapping.set_index('reporter-code')['economy-142']
    code2econ = {int(k): v for k, v in map_series.dropna().to_dict().items() if isinstance(v, str) and v.strip()}
    
    # map rows/cols
    row_labels = [code2econ.get(int(c), None) for c in lm.index]
    col_labels = [code2econ.get(int(c), None) for c in lm.columns.astype(int)]
    row_keep_idx = [i for i,l in enumerate(row_labels) if l is not None]
    col_keep_idx = [i for i,l in enumerate(col_labels) if l is not None]
    lm2 = lm.iloc[row_keep_idx, col_keep_idx]
    rows_mapped = [row_labels[i] for i in row_keep_idx]
    cols_mapped = [col_labels[i] for i in col_keep_idx]
    
    # aggregate duplicates
    agg_rows = lm2.groupby(pd.Index(rows_mapped)).sum()
    agg_full = agg_rows.T.groupby(pd.Index(cols_mapped)).sum().T
    
    # reindex to canonical order
    mat142 = agg_full.reindex(index=target_order, columns=target_order, fill_value=0.0)
    return mat142

In [ ]:
for year in range(1962, 2023):
    input_path = RESULTS / 'trade-data' / 'raw-data' / 'limestone_matrix' / f'limestone_matrix_{year}.csv'
    lm = pd.read_csv(input_path, header=0, index_col=0)
    mat142 = aggregate_matrix_to_142(lm)
    output_path = RESULTS / 'trade-data' / 'trade-142' / f'limestone-matrix-{year}.csv'
    mat142.to_csv(output_path)

In [ ]:
for year in range(1962, 2023):
    input_path = RESULTS / 'trade-data' / 'raw-data' / 'gravel_matrix' / f'gravel_crushed_stone_matrix_{year}.csv'
    lm = pd.read_csv(input_path, header=0, index_col=0)
    mat142 = aggregate_matrix_to_142(lm)
    output_path = RESULTS / 'trade-data' / 'trade-142' / f'gravel-crushed-stone-matrix-{year}.csv'
    mat142.to_csv(output_path)

In [ ]:
for year in range(1988, 2023):
    input_path = RESULTS / 'trade-data' / 'raw-data' / 'sand_matrix' / f'other_sand_matrix_{year}.csv'
    lm = pd.read_csv(input_path, header=0, index_col=0)
    mat142 = aggregate_matrix_to_142(lm)
    output_path = RESULTS / 'trade-data' / 'trade-142' / f'other-sand-matrix-{year}.csv'
    mat142.to_csv(output_path)

In [ ]:
for year in range(1962, 2000):
    input_path = RESULTS / 'trade-data' / 'raw-data' / 'sand_matrix' / f'sand_matrix_{year}.csv'
    lm = pd.read_csv(input_path, header=0, index_col=0)
    mat142 = aggregate_matrix_to_142(lm)
    output_path = RESULTS / 'trade-data' / 'trade-142' / f'sand-matrix-{year}.csv'
    mat142.to_csv(output_path)

### compile to economy level

In [ ]:
limestone_export_142 = pd.DataFrame([], columns=economies_142)
limestone_import_142 = pd.DataFrame([], columns=economies_142)

for year in range(1931, 2023):
    limestone_export_142.loc[year, :] = 0
    limestone_import_142.loc[year, :] = 0

for year in range(1962, 2023):
    input_path = RESULTS / 'trade-data' / 'trade-142' / f'limestone-matrix-{year}.csv'
    trade_matrix = pd.read_csv(input_path, header=0, index_col=0)
    total_export = trade_matrix.sum(axis=1)[economies_142]
    total_import = trade_matrix.sum(axis=0)[economies_142]
    limestone_export_142.loc[year, :] = total_export / 1e6   # unit: kt
    limestone_import_142.loc[year, :] = total_import / 1e6   # unit: kt


In [ ]:
gravel_crushed_stone_export_142 = pd.DataFrame([], columns=economies_142)
gravel_crushed_stone_import_142 = pd.DataFrame([], columns=economies_142)

for year in range(1931, 2023):
    gravel_crushed_stone_export_142.loc[year, :] = 0
    gravel_crushed_stone_import_142.loc[year, :] = 0

for year in range(1962, 2023):
    input_path = RESULTS / 'trade-data' / 'trade-142' / f'gravel-crushed-stone-matrix-{year}.csv'
    trade_matrix = pd.read_csv(input_path, header=0, index_col=0)
    total_export = trade_matrix.sum(axis=1)[economies_142]
    total_import = trade_matrix.sum(axis=0)[economies_142]
    gravel_crushed_stone_export_142.loc[year, :] = total_export / 1e6   # unit: kt
    gravel_crushed_stone_import_142.loc[year, :] = total_import / 1e6   # unit: kt


In [ ]:
other_sand_export_142 = pd.DataFrame([], columns=economies_142)
other_sand_import_142 = pd.DataFrame([], columns=economies_142)

for year in range(1931, 2023):
    other_sand_export_142.loc[year, :] = 0
    other_sand_import_142.loc[year, :] = 0

for year in range(1988, 2023):
    input_path = RESULTS / 'trade-data' / 'trade-142' / f'other-sand-matrix-{year}.csv'
    trade_matrix = pd.read_csv(input_path, header=0, index_col=0)
    total_export = trade_matrix.sum(axis=1)[economies_142]
    total_import = trade_matrix.sum(axis=0)[economies_142]
    other_sand_export_142.loc[year, :] = total_export / 1e6   # unit: kt
    other_sand_import_142.loc[year, :] = total_import / 1e6   # unit: kt
    

In [ ]:
sand_export_142 = pd.DataFrame([], columns=economies_142)
sand_import_142 = pd.DataFrame([], columns=economies_142)

for year in range(1931, 2023):
    sand_export_142.loc[year, :] = 0
    sand_import_142.loc[year, :] = 0

for year in range(1962, 1989):
    input_path = RESULTS / 'trade-data' / 'trade-142' / f'sand-matrix-{year}.csv'
    trade_matrix = pd.read_csv(input_path, header=0, index_col=0)
    total_export = trade_matrix.sum(axis=1)[economies_142]
    total_import = trade_matrix.sum(axis=0)[economies_142]
    sand_export_142.loc[year, :] = total_export / 1e6   # unit: kt
    sand_import_142.loc[year, :] = total_import / 1e6   # unit: kt
    

### output final results

In [ ]:
limestone_export_142.to_csv(RESULTS /'trade-data' / 'limestone-export-142.csv')
limestone_import_142.to_csv(RESULTS /'trade-data' / 'limestone-import-142.csv')

gravel_crushed_stone_export_142.to_csv(RESULTS /'trade-data' / 'gravel-crushed-stone-export-142.csv')
gravel_crushed_stone_import_142.to_csv(RESULTS /'trade-data' / 'gravel-crushed-stone-import-142.csv')

sand_export_142.to_csv(RESULTS /'trade-data' / 'sand-export-142.csv')
sand_import_142.to_csv(RESULTS /'trade-data' / 'sand-import-142.csv')

other_sand_export_142.to_csv(RESULTS /'trade-data' / 'other-sand-export-142.csv')
other_sand_import_142.to_csv(RESULTS /'trade-data' / 'other-sand-import-142.csv')